# COPD 步態資料處理

此筆記串接整合模組。可在本機 Jupyter 或 Colab 執行。預設使用合成座標，圖表不代表病患結果。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'utils/data_processor.py').exists():
    target = root / 'copd-gait'
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/hhsjd938/Analysis-of-walking-posture-in-COPD-patients.git', str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
from examples.make_demo import generate
generate('data/demo')


## 特徵與四分位數

使用個人資料時，改為已確認 COCO 17 點配置的單一對象 CSV 路徑。

In [ ]:
from analyze import export_analysis
summary = export_analysis('data/demo/synthetic_pose.csv', 'results/demo')
summary


In [ ]:
from IPython.display import Image, display
display(Image(filename='results/demo/feature_boxplots.png'))
display(Image(filename='results/demo/feature_traces.png'))


## 分類流程煙霧測試

只確認流程可執行。真實實驗需準備受試者分組及標籤，使用足夠訓練輪數，另行設計驗證；不可將下列合成結果作為模型效能。

In [ ]:
from train import train
metrics = train('data/demo/manifest.csv', output='artifacts/demo', epochs=2)
metrics['mlp']


## Random Forest 的 SHAP

SHAP 說明的是隨機森林的特徵貢獻，不是 MLP 或因果關係。

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-explain.txt'], check=True)
from explain import explain
explain('artifacts/demo/random_forest.joblib', 'artifacts/demo/test_features.csv', 'results/demo/shap')
display(Image(filename='results/demo/shap/shap_importance.png'))
